# GSOM Idionomic Clustering — Colab

Cluster **any matrix of features keyed by an ID** with a Growing Self-Organizing Map.

Powered by [`gsom-idionomic`](https://github.com/ozziejoe/gsom-idionomic). Run each cell top to bottom (`Shift+Enter`).

## 1. Install

In [ ]:
!pip install -q git+https://github.com/ozziejoe/gsom-idionomic.git
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from gsom_idionomic import (GsomConfig, build_map, cluster_map,
                            make_zoo_dataset, make_sample_dataset)
from gsom_idionomic.demo import ZOO_RECOMMENDED, SAMPLE_RECOMMENDED
from gsom_idionomic import plots
print('Ready.')

## 2. Load data
Pick a built-in example, or upload your own CSV (`ID` + any numeric feature columns).
- **zoo** — 101 animals x 16 traits; groups into animal kinds (spread 0.4, K 5).
- **ema** — idionomic person-types with `se_` (shows I²; spread 0.6, K 4).

In [ ]:
DATASET = 'zoo'  #@param ["zoo", "ema", "upload"]
ID_COL = 'ID'    #@param {type:"string"}

if DATASET=='zoo':
    df = make_zoo_dataset(); REC = ZOO_RECOMMENDED; truth = df.attrs['true_class']
elif DATASET=='ema':
    df = make_sample_dataset(); REC = SAMPLE_RECOMMENDED; truth = df.attrs['true_class']
else:
    from google.colab import files
    up=files.upload(); df=pd.read_csv(list(up.keys())[0]); REC={'spread':'auto','k':'auto','sil_cut':0.20}; truth={}
print(df.shape); df.head()

## 3. Step 1 — Build the map
Spread is chosen from map fit only (quantization error + topology). The map is useful on its own — no clustering needed.

In [ ]:
cfg = GsomConfig(id_col=ID_COL, sil_cut=REC['sil_cut'])
m = build_map(df, cfg, spread=REC['spread'])
print(f'spread={m.spread} | active nodes={m.n_nodes} | winner nodes={m.n_winner_nodes}')
display(plots.plot_node_map(m.gsom_map, m.df_map))
if m.df_spread is not None: display(plots.plot_spread_tuning(m.df_spread, m.spread))

## 4. Step 2 — Cluster the map (optional)

In [ ]:
c = cluster_map(m, cfg, k=REC['k'])
print(f'K={c.best_k} | clusters={c.valid_clusters} | outliers={(c.df_clusters.cluster==-1).sum()}')
display(plots.plot_skeleton_map(m.gsom_map, c.df_active, m.df_map, c.df_profiles, ID_COL))
display(plots.plot_cluster_map(c.df_active, m.df_map))
display(plots.plot_silhouette(c.df_active, cfg.sil_cut))
for cl in c.valid_clusters: display(plots.plot_divergence(cl, c, cfg))
if any(col.startswith('se_') for col in df.columns): display(plots.plot_homogeneity_gain(c, cfg))

## 5. Recovery check (built-in examples)
Known group x GSOM cluster — the GSOM never saw the labels.

In [ ]:
if truth:
    xt=c.df_clusters.copy(); xt['Known group']=xt['ID'].map(truth)
    display(pd.crosstab(xt['Known group'], xt['cluster']))

## 6. Tables & download

In [ ]:
display(c.df_summary_table)
import zipfile, os
os.makedirs('out', exist_ok=True)
c.df_clusters.to_csv('out/cluster_assignments.csv', index=False)
c.df_classification.to_csv('out/classification_2x2.csv', index=False)
c.df_full_table.to_csv('out/cluster_table_full.csv', index=False)
with zipfile.ZipFile('gsom_results.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir('out'): zf.write('out/'+f)
from google.colab import files; files.download('gsom_results.zip')